# ASC Phase 1 — Tạo pseudo sentiment labels trên dataset lớn

**Input:** ATE Phase 1 high confidence output (rows với `len(aspects) >= 1`)

**Model:** `asc_teacher_phase1` (RoBERTa-base fine-tuned với aspect-marker format)

**Pipeline:**

1. Format aspect-marker: `The [ASP] screen [/ASP] is dim.`
2. ASC inference → [neg, neu, pos] probabilities per aspect
3. Lọc high/low confidence theo ngưỡng


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os


SPLIT_ID = 0
N_SPLITS = 10

BASE_DIR    = "/content/drive/MyDrive/outputs_electronics_cleaning"
SCRIPT_DIR  = "/content/drive/MyDrive/outputs_electronics_cleaning"
SPLITS_DIR  = f"{BASE_DIR}/split_assignment"
ATE_DIR     = f"{BASE_DIR}/ATE_Phase_1/high"

HIGH_DIR    = f"{BASE_DIR}/ASC_PHASE_1/high"
LOW_DIR     = f"{BASE_DIR}/ASC_PHASE_1/low"
REPORT_DIR  = f"{BASE_DIR}/ASC_PHASE_1/reports/asc_1"

# Thresholds 
POLAR_THRESHOLD   = 0.9
NEUTRAL_THRESHOLD = 0.55

# Inference params 
MAX_LENGTH = 192
CHUNK_SIZE = 50_000  

os.makedirs(HIGH_DIR,   exist_ok=True)
os.makedirs(LOW_DIR,    exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print(f"SPLIT_ID   : {SPLIT_ID} / {N_SPLITS}")
print(f"ATE input  : {ATE_DIR}")
print(f"ASC high   : {HIGH_DIR}")
print(f"ASC low    : {LOW_DIR}")
print(f"Report dir : {REPORT_DIR}")


SPLIT_ID   : 0 / 10
ATE input  : /content/drive/MyDrive/outputs_electronics_cleaning/ATE_Phase_1/high
ASC high   : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_1/high
ASC low    : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_1/low
Report dir : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_1/reports/asc_1


In [3]:
import sys
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)

from analyze_aspect_sentiment import predict_aspect_sentiment

print("analyze_aspect_sentiment loaded OK")

Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

analyze_aspect_sentiment loaded OK


In [ ]:
import pandas as pd

category   = "Electronics_part1"
input_path = f"{ATE_DIR}/high_confidence_{SPLIT_ID:02d}_{category}.parquet"

assert os.path.exists(input_path), (
    f"Không tìm thấy: {input_path}\n"
    f"→ Kiểm tra lại ATE_DIR và SPLIT_ID"
)

split_entries = [{"category": category, "files": [input_path]}]

print(f"Split {SPLIT_ID}/{N_SPLITS} — 1 category:")
print(f"  {category}: {input_path}")

Split 0/10 — 1 category:
  Electronics_part1: /content/drive/MyDrive/outputs_electronics_cleaning/ATE_Phase_1/high/high_confidence_00_Electronics_part1.parquet


In [ ]:
import numpy as np


def write_report(category, n_ate_total, n_with_asp, high_df, low_df):
    """Ghi thống kê ASC phase 1 ra file báo cáo."""
    n_high = len(high_df)
    n_low  = len(low_df)
    high_rate = n_high / n_with_asp * 100 if n_with_asp > 0 else 0.0

    # Tỉ lệ pos/neu/neg theo dominant class mỗi aspect
    n_pos = n_neu = n_neg = total_asp = 0
    if n_high > 0 and "sentiments" in high_df.columns:
        for sents in high_df["sentiments"]:
            for tri in sents:
                total_asp += 1
                dom = int(np.argmax(tri))  
                if dom == 0:   n_neg += 1
                elif dom == 1: n_neu += 1
                else:          n_pos += 1

    def pct(x): return f"{x / total_asp * 100:.2f}%" if total_asp > 0 else "N/A"

    lines = [
        "=" * 60,
        f"ASC Phase 1 Report — {category}",
        "=" * 60,
        "",
        "[Input — ATE High Confidence]",
        f"  Total ATE high conf      : {n_ate_total:>10,}",
        f"  With aspects (processed) : {n_with_asp:>10,}",
        f"  Without aspects (skipped): {n_ate_total - n_with_asp:>10,}",
        "",
        "[ASC Confidence Split]",
        f"  ASC High confidence : {n_high:>10,}  ({high_rate:.2f}% of with_aspects)",
        f"  ASC Low  confidence : {n_low:>10,}  ({100 - high_rate:.2f}% of with_aspects)",
        "",
        "[Thresholds]",
        f"  polar_threshold   : {POLAR_THRESHOLD}",
        f"  neutral_threshold : {NEUTRAL_THRESHOLD}",
        "",
        "[Sentiment Distribution — ASC High Confidence]",
        f"  Total aspects classified : {total_asp:>10,}",
        f"  Positive : {n_pos:>8,}  ({pct(n_pos)})",
        f"  Neutral  : {n_neu:>8,}  ({pct(n_neu)})",
        f"  Negative : {n_neg:>8,}  ({pct(n_neg)})",
        "",
        "[Output files]",
        f"  high: {HIGH_DIR}/high_confidence_{SPLIT_ID:02d}_{category}.parquet",
        f"  low : {LOW_DIR}/low_confidence_{SPLIT_ID:02d}_{category}.parquet",
        "=" * 60,
    ]

    report_path = f"{REPORT_DIR}/asc_1_report_{SPLIT_ID:02d}_{category}.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"  Report saved : {report_path}")
    print(f"  high={n_high:,}  low={n_low:,}  high_rate={high_rate:.1f}%")
    print(f"  aspects → pos={n_pos:,} neu={n_neu:,} neg={n_neg:,}")


print("Helper functions defined.")


Helper functions defined.


In [ ]:
import torch
import gc
import queue
import threading
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as pad
from tqdm.auto import tqdm


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    _vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {_vram_gb:.1f} GB")
    ASC_BATCH_SIZE = 1024 if _vram_gb >= 40 else 512 if _vram_gb >= 15 else 256
else:
    ASC_BATCH_SIZE = 32
    print("WARNING: GPU không khả dụng, tốc độ sẽ rất chậm")

print(f"ASC batch size : {ASC_BATCH_SIZE}")
print(f"CHUNK_SIZE     : {CHUNK_SIZE:,}")


def _iter_chunks(parquet_path, chunk_size, prefetch=3):
    """Stream parquet file theo chunks, prefetch trên CPU thread riêng."""
    ds = pad.dataset(parquet_path, format="parquet")
    q  = queue.Queue(maxsize=prefetch)

    def _producer():
        try:
            for batch in ds.scanner(batch_size=chunk_size).to_batches():
                q.put(batch.to_pandas())
                del batch
        finally:
            q.put(None)

    threading.Thread(target=_producer, daemon=True).start()
    while True:
        item = q.get()
        if item is None:
            break
        yield item


HIGH_COLS = [
    "parent_asin", "sentence_id", "sentence_text",
    "rating", "category_name", "gate_confidence",
    "aspects", "sentiments",
]
LOW_COLS = [
    "parent_asin", "sentence_id", "sentence_text",
    "rating", "category_name", "aspects",
]


for entry in split_entries:
    category   = entry["category"]
    input_path = f"{ATE_DIR}/high_confidence_{SPLIT_ID:02d}_Electronics_part1.parquet"
    out_high   = f"{HIGH_DIR}/high_confidence_{SPLIT_ID:02d}_{category}.parquet"
    out_low    = f"{LOW_DIR}/low_confidence_{SPLIT_ID:02d}_{category}.parquet"

    if not os.path.exists(input_path):
        print(f"[SKIP] ATE input not found: {input_path}")
        continue

    if os.path.exists(out_high) and os.path.exists(out_low):
        print(f"[SKIP] Already done: {category}")
        continue

    ds       = pad.dataset(input_path, format="parquet")
    n_total  = ds.count_rows()
    n_chunks = (n_total + CHUNK_SIZE - 1) // CHUNK_SIZE

    print(f"\n{'─' * 55}")
    print(f" Processing : {category}  ({n_total:,} ATE high conf rows)")
    print(f" SPLIT_ID   : {SPLIT_ID}/{N_SPLITS}")

    writer_high = writer_low = None
    total_high = total_low = actual_with_asp = 0

    try:
        for chunk in tqdm(
            _iter_chunks(input_path, CHUNK_SIZE),
            total=n_chunks,
            desc=f"  split{SPLIT_ID:02d}/{category}",
        ):
            actual_with_asp += (chunk["aspects"].apply(len) >= 1).sum()

            high_df, low_df = predict_aspect_sentiment(
                df=chunk,
                polar_threshold=POLAR_THRESHOLD,
                neutral_threshold=NEUTRAL_THRESHOLD,
                batch_size=ASC_BATCH_SIZE,
                max_length=MAX_LENGTH,
            )
            del chunk

            high_table = pa.Table.from_pandas(high_df[HIGH_COLS], preserve_index=False)
            low_table  = pa.Table.from_pandas(low_df[LOW_COLS],   preserve_index=False)

            if writer_high is None:
                writer_high = pq.ParquetWriter(out_high, high_table.schema)
                writer_low  = pq.ParquetWriter(out_low,  low_table.schema)

            writer_high.write_table(high_table)
            writer_low.write_table(low_table)

            total_high += len(high_df)
            total_low  += len(low_df)

            del high_df, low_df, high_table, low_table
            gc.collect()

    finally:
        if writer_high: writer_high.close()
        if writer_low:  writer_low.close()

    print(f"  Saved high : {out_high}  ({total_high:,} rows)")
    print(f"  Saved low  : {out_low}  ({total_low:,} rows)")

    high_df_final = pd.read_parquet(out_high)
    low_df_final  = pd.read_parquet(out_low)
    write_report(category, n_total, actual_with_asp, high_df_final, low_df_final)
    del high_df_final, low_df_final
    gc.collect()


print(f"\n{'═' * 55}")
print(f" Split {SPLIT_ID} done.")


Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB
ASC batch size : 512
CHUNK_SIZE     : 50,000

───────────────────────────────────────────────────────
 Processing : Electronics_part1  (2,809,361 ATE high conf rows)
 SPLIT_ID   : 0/10


  split00/Electronics_part1:   0%|          | 0/57 [00:00<?, ?it/s]

  Saved high : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_1/high/high_confidence_00_Electronics_part1.parquet  (1,297,541 rows)
  Saved low  : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_1/low/low_confidence_00_Electronics_part1.parquet  (1,511,820 rows)
  Report saved : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_1/reports/asc_1/asc_1_report_00_Electronics_part1.txt
  high=1,297,541  low=1,511,820  high_rate=81.2%
  aspects → pos=1,161,030 neu=2,156 neg=460,682

═══════════════════════════════════════════════════════
 Split 0 done.


In [ ]:
# Summary table
summary_rows = []
for entry in split_entries:
    category  = entry["category"]
    high_path = f"{HIGH_DIR}/high_confidence_{SPLIT_ID:02d}_{category}.parquet"
    low_path  = f"{LOW_DIR}/low_confidence_{SPLIT_ID:02d}_{category}.parquet"

    if not os.path.exists(high_path):
        print(f"[MISSING] {high_path}")
        continue

    high = pd.read_parquet(high_path)
    low  = pd.read_parquet(low_path)

    n_pos = n_neu = n_neg = total_asp = 0
    for sents in high["sentiments"]:
        for tri in sents:
            total_asp += 1
            dom = int(np.argmax(tri))
            if dom == 0:   n_neg += 1
            elif dom == 1: n_neu += 1
            else:          n_pos += 1

    n_total = len(high) + len(low)
    summary_rows.append({
        "category"      : category,
        "split"         : SPLIT_ID,
        "asc_high"      : len(high),
        "asc_low"       : len(low),
        "high_rate_%"   : round(len(high) / n_total * 100, 2) if n_total > 0 else 0,
        "total_aspects" : total_asp,
        "pos_%"         : round(n_pos / total_asp * 100, 2) if total_asp > 0 else 0,
        "neu_%"         : round(n_neu / total_asp * 100, 2) if total_asp > 0 else 0,
        "neg_%"         : round(n_neg / total_asp * 100, 2) if total_asp > 0 else 0,
    })

summary_df = pd.DataFrame(summary_rows)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
print(summary_df.to_string(index=False))


         category  split  asc_high  asc_low  high_rate_%  total_aspects  pos_%  neu_%  neg_%
Electronics_part1      0   1297541  1511820        46.19        1623868   71.5   0.13  28.37
